In [2]:
import os
import pathlib
import json
import pandas as pd
from openai import OpenAI

client = OpenAI()

In [4]:
# gpt-5-nano 모델 호출 예시
def ask_direct(question: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"{question} 최종 답만 숫자로 출력하라."}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(ask_direct("이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"))
# → "213000"   ← 틀림! (정답은 213300)

213300


# (3) 해결책 — "단계적으로 풀어라"

In [5]:
# OpenAI SDK (gpt-5-nano) CoT 적용 예시
def ask_cot(question: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"{question} 생각 과정을 단계별로 작성한 뒤 마지막에 정답을 출력하라."}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(ask_cot("이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"))

1. **이어버드 가격 확인**: 이어버드 하나의 가격은 79,000원입니다.

2. **구매 개수 확인**: 이어버드를 3개 구매했습니다.

3. **총 가격 계산**: 
   - 총 가격 = 이어버드 가격 × 구매 개수
   - 총 가격 = 79,000원 × 3 = 237,000원

4. **쿠폰 할인율 확인**: 10% 쿠폰을 받았습니다.

5. **할인 금액 계산**: 
   - 할인 금액 = 총 가격 × 할인율
   - 할인 금액 = 237,000원 × 0.10 = 23,700원

6. **최종 결제액 계산**: 
   - 최종 결제액 = 총 가격 - 할인 금액
   - 최종 결제액 = 237,000원 - 23,700원 = 213,300원

따라서, 총 결제액은 **213,300원**입니다.


cot = Chain Of Thought

# CoT 함수

In [6]:
def ask_cot(question: str) -> str:
    """CoT: 한 줄씩 풀이를 쓰게 한다.

    [왜] 모델은 앞서 쓴 자기 출력을 다시 입력으로 참고한다. 풀이를 글로 쓰게 하면
    그 풀이가 다음 토큰 생성의 '작업 공간(근거)'이 되어 마지막 답이 정확해진다.
    """
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"{question}\n"
                    "단계적으로 풀어라. 각 계산을 한 줄씩 쓰고, "
                    "맨 마지막 줄에 '정답: <숫자>' 형식으로 답하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [7]:
print(ask_direct("이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"))

213300


# 답에서 숫자만 뽑아내기

In [12]:
import re
from openai import OpenAI


client = OpenAI()
OPENAI_MODEL = 'gpt-4o-mini'

def extract_number(text: str):
    """답변에서 마지막 숫자를 정수로 반환(콤마 제거)."""
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    if not nums:
        return None
    return int(nums[-1].replace(",", ""))

In [14]:
q = "이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"
print("[직접] ", ask_direct(q))
print("[CoT]\n", extract_number(ask_cot(q)))

[직접]  213300
[CoT]
 213300


# self-consistency

In [15]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

def verify(question: str, cot_answer: str) -> str:
    """제출한 풀이를 모델에게 다시 검산시켜 신뢰도를 높인다."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"[문제] {question}\n[제출한 풀이]\n{cot_answer}\n\n"
                    "위 풀이가 맞는지 다시 계산해 검산하라. "
                    "틀렸다면 올바른 값을, 맞다면 그대로 '정답: <숫자>'로 확정하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [17]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

def verify(question: str, cot_answer: str) -> str:
    """제출한 풀이를 모델에게 다시 검산시켜 신뢰도를 높인다."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"[문제] {question}\n[제출한 풀이]\n{cot_answer}\n\n"
                    "위 풀이가 맞는지 다시 계산해 검산하라. "
                    "틀렸다면 올바른 값을, 맞다면 그대로 '정답: <숫자>'로 확정하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [18]:
q = "이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"
first = ask_cot(q)          # 1차 풀이
print("[검산 결과]\n", verify(q, first))   # 2차 검산

[검산 결과]
 제출한 풀이를 검산해보겠습니다.

1. 이어버드 한 개의 가격: 79,000원  
2. 이어버드 3개의 가격: 79,000원 × 3 = 237,000원  
3. 10% 쿠폰 할인액: 237,000원 × 0.10 = 23,700원  
4. 총 결제액: 237,000원 - 23,700원 = 213,300원  

계산이 모두 정확합니다. 

정답: 213300


# (1) 공통 시작부 + 데이터 로드

In [20]:
import os
import pathlib
import re
import pandas as pd
from openai import OpenAI

# OpenAI 클라이언트 초기화 및 모델 설정
client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

DATA_PATH = pathlib.Path("./data")
# math_word_problems.csv = 쇼핑 계산 문제 8개. answer 컬럼이 정수 정답.
df = pd.read_csv(DATA_PATH / "math_word_problems.csv")

print("문제 수:", len(df))
print(df.iloc[0]["question"], "→ 정답:", df.iloc[0]["answer"])

문제 수: 8
승승장구몰에서 이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은? → 정답: 213300


## (2) 세 함수 준비

02번에서 만든 `ask_direct`, `ask_cot`, `extract_number`를 그대로 씁니다.

In [22]:
def extract_number(text: str):
    """답변에서 마지막 숫자를 정수로 반환(콤마 제거)."""
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    return int(nums[-1].replace(",", "")) if nums else None

def ask_direct(question: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 최종 숫자(원)만 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

def ask_cot(question: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"{question}\n"
                    "단계적으로 풀어라. 각 계산을 한 줄씩 쓰고, "
                    "맨 마지막 줄에 '정답: <숫자>' 형식으로 답하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [23]:
direct_ok = cot_ok = 0
for _, row in df.iterrows():
    ans = int(row["answer"])
    d = extract_number(ask_direct(row["question"]))   # 직접 답변의 숫자
    c = extract_number(ask_cot(row["question"]))      # CoT 답변의 숫자
    direct_ok += (d == ans)     # bool(True/False)은 1/0으로 더해진다
    cot_ok += (c == ans)
    print(f"{row['problem_id']} 정답={ans:>7} | 직접={d} {'O' if d==ans else 'X'}"
          f" | CoT={c} {'O' if c==ans else 'X'}")

n = len(df)
print(f"\n직접 답변 정답률 : {direct_ok}/{n} = {direct_ok/n:.0%}")
print(f"CoT  정답률      : {cot_ok}/{n} = {cot_ok/n:.0%}")

M01 정답= 213300 | 직접=213000 X | CoT=213300 O
M02 정답= 386650 | 직접=425350 X | CoT=386650 O
M03 정답= 107000 | 직접=111000 X | CoT=107000 O
M04 정답=  53000 | 직접=53000 O | CoT=53000 O
M05 정답= 128800 | 직접=25600 X | CoT=128800 O
M06 정답=   5320 | 직접=5320 O | CoT=5320 O
M07 정답=  94400 | 직접=94400 O | CoT=94400 O
M08 정답= 351000 | 직접=351000 O | CoT=351000 O

직접 답변 정답률 : 4/8 = 50%
CoT  정답률      : 8/8 = 100%


# 5.6 CoT가 효과 없는 경우

### (2) (a) 단순 질문 — 토큰 낭비

In [25]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

def ask_direct(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 정답만 한 단어로 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens

def ask_cot(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n단계적으로 풀어라. 마지막 줄에 '정답: <값>'."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens

d_txt, d_tok = ask_direct("대한민국의 수도는 어디인가?")
c_txt, c_tok = ask_cot("대한민국의 수도는 어디인가?")
print(f"직접: {d_txt} (토큰 {d_tok})")
print(f"CoT : 토큰 {c_tok}  ← 같은 정답인데 토큰만 더 씀")

직접: 서울 (토큰 30)
CoT : 토큰 115  ← 같은 정답인데 토큰만 더 씀


In [26]:
cases = [
    ("배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?", "50"),
    ("5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?", "5"),  # 정답 5분
]

for q, gold in cases:
    d_txt, _ = ask_direct(q)
    c_txt, _ = ask_cot(q)
    print(f"Q: {q}")
    print(f"  정답: {gold} / 직접: {d_txt[:20]} / CoT 마지막줄: {c_txt.splitlines()[-1]}")

Q: 배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?
  정답: 50 / 직접: 100원 / CoT 마지막줄: 정답: 50
Q: 5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?
  정답: 5 / 직접: 5 / CoT 마지막줄: 정답: 5


#  (b) 함정 문제 — CoT도 흔들린다

In [27]:
import os
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

def ask_direct(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 정답만 한 단어로 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens

def ask_cot(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n단계적으로 풀어라. 마지막 줄에 '정답: <값>'."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens


In [28]:
cases = [
    ("배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?", "50"),
    ("5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?", "5"),  # 정답 5분
]

for q, gold in cases:
    d_txt, _ = ask_direct(q)
    c_txt, _ = ask_cot(q)
    print(f"Q: {q}")
    print(f"  정답: {gold} / 직접: {d_txt[:20]} / CoT 마지막줄: {c_txt.splitlines()[-1]}")

Q: 배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?
  정답: 50 / 직접: 100원 / CoT 마지막줄: 정답: 50
Q: 5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?
  정답: 5 / 직접: 5 / CoT 마지막줄: 정답: 5


# (4) 토큰 총량 비교

In [29]:
####################################################
# 토큰 합산용 변수 초기화
direct_tok_sum = 0
cot_tok_sum = 0
cases = [
    ("배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?", "50"),
    ("5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?", "5"),  # 정답 5분
]

for q, gold in cases:
    d_txt, d_tok = ask_direct(q)
    c_txt, c_tok = ask_cot(q)
    
    # 토큰 수 누적
    direct_tok_sum += d_tok
    cot_tok_sum += c_tok
    
    last_line_cot = c_txt.splitlines()[-1] if c_txt.splitlines() else c_txt
    print(f"Q: {q}")
    print(f"  정답: {gold} / 직접: {d_txt[:20]} / CoT 마지막줄: {last_line_cot}")

print("-" * 50)


Q: 배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?
  정답: 50 / 직접: 100원 / CoT 마지막줄: 정답: 50
Q: 5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?
  정답: 5 / 직접: 5 / CoT 마지막줄: 정답: 5
--------------------------------------------------


In [30]:
# (전체 케이스의 토큰을 합산)
print(f"총 토큰 — 직접: {direct_tok_sum} / CoT: {cot_tok_sum} "
      f"(CoT가 {cot_tok_sum - direct_tok_sum}토큰 더 사용)")

총 토큰 — 직접: 105 / CoT: 580 (CoT가 475토큰 더 사용)


# 문제 2 — self-consistency 다중 풀이

In [31]:
import pathlib
import re
from collections import Counter
import pandas as pd
from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

DATA_PATH = pathlib.Path("./data")
df = pd.read_csv(DATA_PATH / "math_word_problems.csv")

def extract_number(text):
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    return int(nums[-1].replace(",", "")) if nums else None

def self_consistency(question: str, n: int = 3):
    """같은 문제를 n번 풀어 가장 많이 나온 답(다수결)을 채택한다."""
    answers = []
    for _ in range(n):
        r = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": f"{question}\n단계적으로 풀고 마지막 줄에 '정답: <숫자>'."
                }
            ],
            temperature=0.7,  # 다양성 위해 약간 높임
        )
        answers.append(extract_number(r.choices[0].message.content))
    print("개별 답:", answers)
    winner = Counter(answers).most_common(1)[0][0]   # 최빈값
    print("다수결 답:", winner)
    return winner

if __name__ == "__main__":
    row = df.iloc[0]
    result = self_consistency(row["question"], n=3)
    print("실제 정답:", int(row["answer"]), "→", "맞음" if result == int(row["answer"]) else "틀림")

개별 답: [213300, 213300, 213300]
다수결 답: 213300
실제 정답: 213300 → 맞음
